# Promoter Signal — Validation Notebook

**Components:** QoQ change (35%), 1yr trend (35%), pledge quality (30%)  
**Special:** Asymmetric selling adjustment, holding level modifier  
**Pass criteria:** QoQ/trend correlation ≥ 0.95. Composite correlation ≥ 0.85.

In [ ]:
import sys, os
os.chdir(os.path.expanduser("~/alpha-signal-v2"))
sys.path.insert(0, ".")

import pandas as pd
import numpy as np
from signals.promoter import _load_data, _compute_scores

stocks, sh = _load_data()
v2 = _compute_scores(stocks, sh)
print(f"v2: {len(v2)} stocks, {v2['promoter_signal'].notna().sum()} with signal")

v1 = pd.read_csv(os.path.expanduser("~/alpha-signal/data/signals/promoter.csv"))
print(f"v1: {len(v1)} stocks")

merged = v2.merge(v1[["sid", "promoter_qoq", "promoter_trend_4q", "promoter_signal"]],
                  on="sid", how="inner", suffixes=("_v2", "_v1"))
print(f"Overlapping: {len(merged)}")

pairs = [
    ("promoter_qoq_v2", "promoter_qoq_v1", "QoQ"),
    ("promoter_signal_v2", "promoter_signal_v1", "Composite"),
]

print(f"\n{'Component':<14} {'Both':>8} {'Corr':>8} {'Mean|diff|':>10}")
print("-" * 44)
for v2c, v1c, label in pairs:
    both = merged[[v2c, v1c]].dropna()
    n = len(both)
    if n > 10:
        corr = both[v2c].corr(both[v1c])
        diff = (both[v2c] - both[v1c]).abs().mean()
        print(f"{label:<14} {n:>8} {corr:>8.4f} {diff:>10.4f}")

## Save to DB (only if validation passed)

In [ ]:
# from signals.promoter import compute
# compute(dry_run=False)